# IMA Plotter – Demo

End-to-end walkthrough: load Excel data → explore → baseline subtraction → interactive plots.

In [1]:
# Cell 1 – Setup & Imports
# Assumes you've run `uv sync --extra notebook` to set up the environment
import os

from ima_plotter import load_excel_files, subtract_baseline, plot_magnetic_vs_time

SAMPLE_DIR = os.path.join("..", "sample_data")

In [2]:
# Cell 2 – Load Data
df = load_excel_files(SAMPLE_DIR)

print(f"Shape: {df.shape}")
df.head()

Shape: (752, 10)


,id,experiment,group,frequency,time_index,avg_v2abs_magnetic,std_v2abs_magnetic,n,avg_time,cv_v2abs_magnetic
0,EX1A,Exp1-on-off-field-summary,Cal0,2,1,0.002072,0.000158,2,0.054159,7.636890
1,EX1A,Exp1-on-off-field-summary,Cal0,2,2,0.002640,0.000159,2,0.255862,6.024507
2,EX1A,Exp1-on-off-field-summary,Cal0,2,3,0.002840,0.000163,2,0.457095,5.733121
3,EX1A,Exp1-on-off-field-summary,Cal0,2,4,0.002978,0.000153,2,0.659203,5.151751
4,EX1A,Exp1-on-off-field-summary,Cal0,2,5,0.003081,0.000156,2,0.860477,5.069891


In [3]:
# Cell 3 – Explore Data
print("IDs       :", sorted(df["id"].unique()))
print("Groups    :", sorted(df["group"].unique()))
print("Frequencies:", sorted(df["frequency"].unique()))
print()
df[["avg_v2abs_magnetic", "std_v2abs_magnetic", "avg_time"]].describe()

IDs       : ['EX1A', 'EX1B', 'EX1C', 'EX1D', 'REF']
Groups    : ['Cal0', 'Cal3']
Frequencies: [np.int64(2), np.int64(7), np.int64(14), np.int64(30)]



,avg_v2abs_magnetic,std_v2abs_magnetic,avg_time
count,752.000000,7.520000e+02,752.000000
mean,0.002759,2.223566e-04,2.898456
std,0.001893,2.223883e-04,1.788178
min,0.000586,2.177114e-07,0.053610
25%,0.001439,1.003840e-04,1.304874
50%,0.002037,1.412499e-04,2.900412
75%,0.003279,2.442805e-04,4.485955
max,0.011141,9.575252e-04,5.962082


In [5]:
# Cell 4 – Baseline Subtraction
# Subtract Cal0 from every group within each id + frequency combination.
# Subtraction is time-matched: Cal0 at each time_index is subtracted from
# all other groups at the same time_index.
# Combined uncertainty: sqrt(std_original² + std_baseline²)
df_sub = subtract_baseline(df, baseline_group="Cal0")

# Filter out Cal0 since it will have delta=0 at every time point after subtraction
df_sub = df_sub[df_sub["group"] != "Cal0"]

# Before vs after for one representative slice
sample = df_sub[(df_sub["id"] == "EX1A") & (df_sub["frequency"] == 2)]
sample[["group", "avg_time", "avg_v2abs_magnetic", "delta_avg_v2abs_magnetic"]].head(8)

,group,avg_time,avg_v2abs_magnetic,delta_avg_v2abs_magnetic
120,Cal3,0.053785,0.002275,0.000203
121,Cal3,0.255045,0.003135,0.000495
122,Cal3,0.457047,0.003615,0.000775
123,Cal3,0.658688,0.004028,0.001049
124,Cal3,0.859918,0.004402,0.001321
125,Cal3,1.061351,0.004763,0.001590
126,Cal3,1.263183,0.005108,0.001858
127,Cal3,1.464571,0.005429,0.002114


In [6]:
# Cell 5 – Basic Plot
# Single frequency, single ID, all groups, with error bars
fig = plot_magnetic_vs_time(
    df_sub,
    filter_frequency=2,
    filter_id="EX1A",
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.show()

In [16]:
# Cell 6 – Facet by ID
# One subplot per sample ID, single frequency
fig = plot_magnetic_vs_time(
    df_sub,
    facet_by=["frequency",None],
    #filter_frequency=2,
    color_by="id",
    use_baseline_subtracted=True,
    show_error_bars=False,
)
fig.update_layout(width=600, height=1200)
fig.show()

In [8]:
# Cell 7 – Facet by Frequency
# One subplot per frequency, single ID
fig = plot_magnetic_vs_time(
    df_sub,
    facet_by="frequency",
    filter_id="EX1A",
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.show()

In [9]:
# Cell 8 – Combined Faceting (ID × Frequency)
# Grid of subplots: rows = ID, columns = frequency
# Filter to a subset to keep the grid manageable
fig = plot_magnetic_vs_time(
    df_sub,
    facet_by=["id", "frequency"],
    filter_id=["EX1A", "EX1B"],
    filter_frequency=[2, 7],
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Cell 9 – Export Data
output_path = os.path.join("..", "processed_data.csv")
df_sub.to_csv(output_path, index=False)
print(f"Saved processed data to: {os.path.abspath(output_path)}")